# 01 — Cleaning, Sentiment & Features

Normalizes data non-destructively, documents row loss, and produces analysis-ready datasets.

In [ ]:
from pathlib import Path
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")
current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / "config" / "config.yaml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate project root containing config/config.yaml")
with (PROJECT_ROOT / "config" / "config.yaml").open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", OUTPUT / "powerbi", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)
fingerprint_paths = [PROJECT_ROOT / "config" / "config.yaml", project_path("shops")]
for key in ("products", "reviews"):
    fingerprint_paths.extend(sorted(project_path(key).glob("shop_*.json")))
seller_candidate = project_path("seller_metrics")
for candidate in (seller_candidate, seller_candidate.with_suffix(".xlsx")):
    if candidate.exists(): fingerprint_paths.append(candidate)
digest = hashlib.sha256()
for path in fingerprint_paths:
    digest.update(str(path.relative_to(PROJECT_ROOT)).encode("utf-8")); digest.update(path.read_bytes())
INPUT_FINGERPRINT = digest.hexdigest()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_FINGERPRINT:", INPUT_FINGERPRINT)

In [ ]:
SHOP_REQUIRED = ["Shop_ID", "Shop_Name", "Shop_type", "Years_Active", "Is_Online_Now", "Total_Products", "Follower_Count_k", "Rating_Average", "Total_Ratings_k", "Chat_Response_Rate_pct", "Has_Voucher", "Target_Label", "Time_Collected"]
PRODUCT_REQUIRED = ["Shop_ID", "product_id", "product_name", "product_details", "description_text", "Review_stars", "units_sold", "product_url", "Time_Collected", "Data_Source"]
REVIEW_REQUIRED = ["Shop_ID", "product_id", "review_id", "user_name", "rating", "review_time", "review_text", "has_image", "source_url", "Time_Collected", "Data_Source", "Verification_Status"]
SELLER_REQUIRED = ["Shop_ID", "Conversion_Rate_pct", "Source", "Collection_Status"]

def load_json_folder(folder, pattern, columns):
    records = []
    for path in sorted(folder.glob(pattern)):
        with path.open(encoding="utf-8") as handle:
            payload = json.load(handle)
        if not isinstance(payload, list):
            raise ValueError(f"{path}: top-level JSON must be a list")
        records.extend(payload)
    return pd.DataFrame(records) if records else pd.DataFrame(columns=columns)

def load_seller_metrics(path):
    candidates = [path, path.with_suffix(".xlsx")]
    source = next((p for p in candidates if p.exists()), None)
    if source is None:
        return pd.DataFrame(columns=SELLER_REQUIRED), None
    frame = pd.read_excel(source, dtype={"Shop_ID": str}) if source.suffix == ".xlsx" else pd.read_csv(source, dtype={"Shop_ID": str})
    return frame, source

shops = pd.read_excel(project_path("shops"), dtype={"Shop_ID": str})
products = load_json_folder(project_path("products"), "shop_*_products.json", PRODUCT_REQUIRED)
reviews = load_json_folder(project_path("reviews"), "shop_*_reviews.json", REVIEW_REQUIRED)
seller_metrics, seller_source = load_seller_metrics(project_path("seller_metrics"))

# Canonicalize legacy/manual collection fields without inventing business values.
manifest_path = PROJECT_ROOT / "src" / "crawl_data" / "product_review_mapping.json"
if manifest_path.exists():
    manifest = pd.DataFrame(json.loads(manifest_path.read_text(encoding="utf-8")))
    if {"product_id", "product_url"} <= set(manifest):
        url_map = manifest.drop_duplicates("product_id").set_index("product_id")["product_url"]
        if "product_url" not in products: products["product_url"] = products["product_id"].map(url_map)
        else: products["product_url"] = products["product_url"].fillna(products["product_id"].map(url_map))
for column in PRODUCT_REQUIRED:
    if column not in products: products[column] = np.nan
if "product_details" in products:
    detail_url = products["product_details"].map(lambda value: value.get("source_product_url") if isinstance(value, dict) else None)
    detail_date = products["product_details"].map(lambda value: value.get("collection_date") if isinstance(value, dict) else None)
    sold_display = products["product_details"].map(lambda value: value.get("public_sold_display") if isinstance(value, dict) else None)

    def parse_public_sold_lower_bound(value):
        """Parse Shopee's public rounded sold display as an observed lower bound."""
        if pd.isna(value):
            return np.nan
        match = __import__("re").search(r"([0-9]+(?:[.,][0-9]+)?)\s*([kKmM]?)", str(value))
        if not match:
            return np.nan
        number = float(match.group(1).replace(",", "."))
        multiplier = {"": 1, "k": 1_000, "m": 1_000_000}[match.group(2).lower()]
        return number * multiplier

    products["product_url"] = products["product_url"].fillna(detail_url)
    products["Time_Collected"] = products["Time_Collected"].fillna(detail_date)
    products["units_sold"] = pd.to_numeric(products["units_sold"], errors="coerce").fillna(sold_display.map(parse_public_sold_lower_bound))
    products["units_sold_display"] = sold_display
    products["units_sold_semantics"] = products["units_sold"].notna().map({True: "public_display_lower_bound", False: None})
products["Data_Source"] = products["Data_Source"].fillna(products["product_url"])
for column in REVIEW_REQUIRED:
    if column not in reviews: reviews[column] = np.nan
if "collection_date" in reviews: reviews["Time_Collected"] = reviews["Time_Collected"].fillna(reviews["collection_date"])
# Reviews were collected from each product's public review endpoint/page.  Link
# provenance through product_id in memory so the immutable raw JSON is not
# rewritten merely to duplicate product-level collection metadata.
product_provenance = products.drop_duplicates("product_id").set_index("product_id")
review_source = reviews["product_id"].map(product_provenance["product_url"])
review_collected = reviews["product_id"].map(product_provenance["Time_Collected"])
reviews["source_url"] = reviews["source_url"].fillna(review_source)
reviews["Time_Collected"] = reviews["Time_Collected"].fillna(review_collected)
reviews["Data_Source"] = reviews["Data_Source"].fillna(reviews["source_url"])
linked = reviews["source_url"].notna() & reviews["Time_Collected"].notna()
reviews.loc[linked & reviews["Verification_Status"].isna(), "Verification_Status"] = "linked_to_product_provenance"
print({"shops": len(shops), "products": len(products), "reviews": len(reviews), "seller_metrics": len(seller_metrics)})

In [ ]:
import html, re, unicodedata

def clean_text(value):
    if value is None or pd.isna(value): return ""
    text = unicodedata.normalize("NFC", html.unescape(str(value))).lower()
    text = re.sub(r"<[^>]+>|https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"([!?.,])\1+", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return re.sub(r"\s+([!?.,])", r"\1", text)

STOPWORDS = {"và","là","của","có","cho","một","những","các","được","với","thì","mà","ở","đã","này","đó","rất"}
def remove_stopwords(value):
    return " ".join(token for token in str(value).split() if token not in STOPWORDS)

shops_clean, products_clean, reviews_clean = shops.copy(), products.copy(), reviews.copy()
reviews_clean = reviews_clean.loc[reviews_clean["review_text"].fillna("").astype(str).str.strip().ne("")].copy()
numeric_shop = ["Years_Active", "Is_Online_Now", "Total_Products", "Follower_Count_k", "Rating_Average", "Total_Ratings_k", "Chat_Response_Rate_pct", "Has_Voucher"]
for col in numeric_shop: shops_clean[col] = pd.to_numeric(shops_clean[col], errors="coerce")
for col in ["Review_stars", "units_sold"]: products_clean[col] = pd.to_numeric(products_clean[col], errors="coerce")
reviews_clean["rating"] = pd.to_numeric(reviews_clean["rating"], errors="coerce")
reviews_clean["has_image"] = pd.to_numeric(reviews_clean["has_image"], errors="coerce")
shops_clean["Time_Collected"] = pd.to_datetime(shops_clean["Time_Collected"], errors="coerce").dt.date.astype("string")
products_clean["Time_Collected"] = pd.to_datetime(products_clean["Time_Collected"], errors="coerce").dt.date.astype("string")
reviews_clean["review_time"] = pd.to_datetime(reviews_clean["review_time"], errors="coerce").dt.date.astype("string")
products_clean["description_normalized"] = products_clean["description_text"].map(clean_text)
reviews_clean["review_normalized"] = reviews_clean["review_text"].map(clean_text)
products_clean["description_clean"] = products_clean["description_normalized"].map(remove_stopwords)
reviews_clean["review_clean"] = reviews_clean["review_normalized"].map(remove_stopwords)
reviews_clean["Comment_Length"] = reviews_clean["review_text"].fillna("").astype(str).str.len()
reviews_clean["Word_Count"] = reviews_clean["review_clean"].str.split().str.len()
reviews_clean["Suspicious_Username"] = reviews_clean["user_name"].fillna("").astype(str).str.match(r"^(user\d{5,}|[a-z]\*{3,}[a-z])$", case=False).astype(int)
products_clean["Description_Length"] = products_clean["description_clean"].str.len()
products_clean["Detail_Field_Count"] = products_clean["product_details"].map(lambda x: len(x) if isinstance(x, dict) else 0)
products_clean["Transparency_Score"] = ((products_clean["Description_Length"].clip(upper=1000) / 1000) * .5 + (products_clean["Detail_Field_Count"].clip(upper=10) / 10) * .5)
observed_sales = products_clean.groupby("Shop_ID", as_index=False)["units_sold"].sum(min_count=1).rename(columns={"units_sold":"Sales"})
shops_clean = shops_clean.drop(columns=["Sales"], errors="ignore").merge(observed_sales, on="Shop_ID", how="left", validate="one_to_one")

In [ ]:
POSITIVE = {"tốt", "đẹp", "ưng", "ok", "ổn", "nhanh", "chuẩn", "tuyệt", "hài lòng", "chất lượng"}
NEGATIVE = {"tệ", "xấu", "chậm", "lỗi", "hỏng", "kém", "thất vọng", "giả", "không tốt", "không hài lòng"}
def sentiment_score(text):
    value = "" if text is None or pd.isna(text) else str(text).lower()
    tokens = set(re.findall(r"\w+", value, flags=re.UNICODE))
    positive = sum(term in value if " " in term else term in tokens for term in POSITIVE)
    negative = sum(term in value if " " in term else term in tokens for term in NEGATIVE)
    return float((positive-negative) / max(positive+negative, 1))
reviews_clean["Sentiment_Score"] = reviews_clean["review_clean"].map(sentiment_score)
reviews_clean["Sentiment_Label"] = pd.cut(reviews_clean["Sentiment_Score"], [-1.01,-0.01,0.01,1.01], labels=["negative","neutral","positive"])

if not seller_metrics.empty:
    seller = seller_metrics.copy()
    seller["Conversion_Rate_pct"] = pd.to_numeric(seller["Conversion_Rate_pct"], errors="coerce")
    shop_seller = seller.groupby("Shop_ID", as_index=False).agg(Conversion_Rate_pct=("Conversion_Rate_pct","mean"))
    shops_clean = shops_clean.drop(columns=["Conversion_Rate_pct"], errors="ignore").merge(shop_seller, on="Shop_ID", how="left", validate="one_to_one")
    if "product_id" in seller:
        products_clean = products_clean.merge(seller[["Shop_ID","product_id","Conversion_Rate_pct"]], on=["Shop_ID","product_id"], how="left", validate="one_to_one")

audit = pd.DataFrame([
    {"dataset":"shops","raw_rows":len(shops),"clean_rows":len(shops_clean),"dropped_rows":len(shops)-len(shops_clean)},
    {"dataset":"products","raw_rows":len(products),"clean_rows":len(products_clean),"dropped_rows":len(products)-len(products_clean)},
    {"dataset":"reviews","raw_rows":len(reviews),"clean_rows":len(reviews_clean),"dropped_rows":len(reviews)-len(reviews_clean)},
])
merged = reviews_clean.merge(products_clean, on=["Shop_ID","product_id"], how="left", validate="many_to_one", suffixes=("_review","_product")).merge(shops_clean, on="Shop_ID", how="left", validate="many_to_one", suffixes=("","_shop"))
if len(merged) != len(reviews_clean): raise AssertionError("Join duplicated or lost review rows")
shops_clean.to_csv(PROCESSED / "shops_clean.csv", index=False)
products_clean.to_csv(PROCESSED / "products_clean.csv", index=False)
reviews_clean.to_csv(PROCESSED / "reviews_clean.csv", index=False)
merged.to_csv(PROCESSED / "merged_dataset.csv", index=False)
audit.to_csv(OUTPUT / "reports" / "cleaning_audit.csv", index=False)
display(audit); display(reviews_clean.head())